In [ ]:
"""
Extrae la serie de evapotranspiración real (ET) quincenal para un punto,
usando MOD16A2GF (MODIS, gap-filled, 500m, Penman-Monteith).

CÓMO LEER EL RESULTADO
-----------------------
et_total_mm    Agua total perdida por evaporación + transpiración en
               esa quincena (según la vegetación actual del pixel, no
               necesariamente el cultivo que planeás poner - ver nota
               abajo).

USO EN EL BALANCE HÍDRICO: junto con precipitation_profile.py (entrada)
y soil_hydraulics.py (capacidad de almacenamiento), esto cierra el
balance:
    Δalmacenamiento = precipitación - ET - excedente (cuando supera AWC)

LIMITACIÓN IMPORTANTE: ET real refleja la vegetación EXISTENTE en el
pixel hoy, no la del cultivo que planees plantar (que puede consumir
más o menos agua). Sirve como piso orientativo para decidir si hace
falta capacidad de riego instalada, no como dimensionamiento exacto
del sistema.

LIMITACIÓN TÉCNICA (fechas): MOD16A2GF es "year-end gap-filled" -> el
relleno de huecos se hace recién al CERRAR cada año completo. Mientras
el año en curso no termine, puede no haber NINGÚN dato disponible para
esos meses (no un rezago de días/semanas como con CHIRPS/ERA5-Land,
sino el año entero hasta que cierre). El script maneja esto devolviendo
NaN para esas quincenas, igual que con los demás rezagos de publicación.

LIMITACIÓN TÉCNICA (composites): MOD16A2GF viene en composites de 8
días, que no calzan exactamente con los bordes de las quincenas (1-15,
16-fin de mes). El total de cada quincena se arma sumando los
composites de 8 días cuya fecha de inicio cae dentro del rango -> puede
haber un pequeño corrimiento en los bordes (unos pocos días de un
composite "prestados" a la quincena vecina). Para el objetivo de
detectar déficit general, esto no afecta la conclusión.
"""
from datetime import date, datetime, timedelta
from pathlib import Path
import pandas as pd
import ee
ee.Initialize()

from period_utils import build_biweekly_periods

def get_et_biweekly(lat, lon, start_date="2016-01-01", end_date=None):
    """
    lat, lon: coordenadas del punto
    start_date, end_date: rango de fechas (str "YYYY-MM-DD"); end_date=None -> hoy
    """
    start = datetime.strptime(start_date, "%Y-%m-%d").date()
    end = datetime.strptime(end_date, "%Y-%m-%d").date() if end_date else date.today()

    periods = build_biweekly_periods(start, end)
    print(f"[DEBUG] {len(periods)} quincenas a procesar, desde {start} hasta {end}")

    point = ee.Geometry.Point([lon, lat])

    # ET viene escalada: valor crudo x 0.1 = mm reales acumulados en el composite
    et_coll = (
        ee.ImageCollection('MODIS/061/MOD16A2GF')
        .select('ET')
        .filterDate(str(start), str(end + timedelta(days=1)))
    )

    ee_periods = ee.List([
        {'label': label, 'start': str(p_start), 'end': str(p_end)}
        for label, p_start, p_end in periods
    ])

    def compute_period(period):
        period = ee.Dictionary(period)
        p_start = ee.Date(period.get('start'))
        p_end = ee.Date(period.get('end'))

        filtered = et_coll.filterDate(p_start, p_end)

        # MOD16A2GF es "year-end gap-filled": el año en curso puede no
        # tener NINGUNA imagen todavía (no solo un rezago de días/semanas
        # como CHIRPS/ERA5, sino el año completo hasta que cierre). Una
        # colección vacía da una imagen de 0 bandas -> hay que armar una
        # imagen enmascarada (sin dato válido) en ese caso, en vez de
        # dejar que falle el multiply/rename.
        has_data = filtered.size().gt(0)
        total = ee.Image(ee.Algorithms.If(
            has_data,
            filtered.sum().multiply(0.1).rename('et_total_mm'),
            ee.Image.constant(0).rename('et_total_mm').selfMask()
        ))

        stats = total.reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=point,
            scale=500,  # resolución nativa de MOD16
            maxPixels=1e9
        )

        return ee.Feature(
            None,
            stats
            .set('label', period.get('label'))
            .set('periodo_inicio', p_start.format('YYYY-MM-dd'))
            .set('periodo_fin', p_end.advance(-1, 'day').format('YYYY-MM-dd'))
        )

    features = ee.FeatureCollection(ee_periods.map(compute_period))
    result = features.getInfo()  # única llamada de red para todos los periodos

    rows = []
    for f in result['features']:
        props = f['properties']
        rows.append({
            'periodo_inicio': props.get('periodo_inicio'),
            'periodo_fin': props.get('periodo_fin'),
            'label': props.get('label'),
            # 'lat': lat,
            # 'lon': lon,
            'et_total_mm': round(props.get('et_total_mm'), 2) if props.get('et_total_mm') is not None else None,
        })

    df = pd.DataFrame(rows)
    df['periodo_inicio'] = pd.to_datetime(df['periodo_inicio'])
    df = df.sort_values('periodo_inicio').reset_index(drop=True)

    filas_nulas = df['et_total_mm'].isna().sum()
    if filas_nulas > 0:
        primeras_nulas = df[df['et_total_mm'].isna()]['label'].tolist()
        print(f"[AVISO] {filas_nulas} quincena(s) sin datos todavía (MOD16A2GF "
              f"rellena huecos recién al cerrar el año -> el año en curso puede "
              f"faltar completo): {primeras_nulas}")

    return df


def save_et_profile(df, out_prefix="et_biweekly", output_dir="../databases"):
    """
    Guarda la serie de evapotranspiración con timestamp en el nombre:
    {out_prefix}-vYYMMDDHHMMSS.csv (mismo patrón que el resto del pipeline)
    """
    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    df.to_csv(out_path, index=False)
    print(f"CSV guardado en {out_path} ({df.shape[0]}x{df.shape[1]})")
    return out_path


if __name__ == "__main__":
    # Sugarcane_QLD
    Latitude, Longitude =-19.689669877950884,147.22717515914223
    
    # El Playon         --||     7.4584221918243045,    -73.222052853104
    # Finca Matanza     --||     7.300921,              -73.009794
    # Sugarcane_COL     --||     3.580109040361371,     -76.31299479308868
    # Sugarcane_QLD     --||     -19.689669877950884,   147.22717515914223

    df = get_et_biweekly(Latitude, Longitude, start_date="2016-01-01")
    out_path = save_et_profile(df)
    print(df.head(10))

[DEBUG] 254 quincenas a procesar, desde 2016-01-01 hasta 2026-08-07
[AVISO] 14 quincena(s) sin datos todavía (MOD16A2GF rellena huecos recién al cerrar el año -> el año en curso puede faltar completo): ['2026-01_Q1', '2026-01_Q2', '2026-02_Q1', '2026-02_Q2', '2026-03_Q1', '2026-03_Q2', '2026-04_Q1', '2026-04_Q2', '2026-05_Q1', '2026-05_Q2', '2026-06_Q1', '2026-06_Q2', '2026-07_Q1', '2026-07_Q2']
CSV guardado en ../databases/et_biweekly-v260807151546.csv (254x6)
  periodo_inicio periodo_fin       label       lat         lon  et_total_mm
0     2016-01-01  2016-01-15  2016-01_Q1 -19.68967  147.227175         25.8
1     2016-01-16  2016-01-31  2016-01_Q2 -19.68967  147.227175         40.4
2     2016-02-01  2016-02-15  2016-02_Q1 -19.68967  147.227175         51.7
3     2016-02-16  2016-02-29  2016-02_Q2 -19.68967  147.227175         41.6
4     2016-03-01  2016-03-15  2016-03_Q1 -19.68967  147.227175         45.3
5     2016-03-16  2016-03-31  2016-03_Q2 -19.68967  147.227175         40.1
6 

In [1]:
"""
Módulo: evapotranspiration_profile.py
Descripción:
    Extrae el perfil de evapotranspiración real (ET) quincenal desde el año 2016 
    hasta la fecha actual para un punto geográfico dado (latitud y longitud), 
    utilizando una estrategia de doble fuente en Google Earth Engine:
    
    1. MODIS/061/MOD16A2GF (Gap-Filled): Fuente primaria con corrección de 
       vacíos al cierre de año.
    2. MODIS/061/MOD16A2: Fuente secundaria (fallback) en tiempo real para 
       cubrir quincenas donde la fuente primaria no posea registros publicados.

CÓMO LEER EL RESULTADO:
    El archivo CSV generado contiene las siguientes columnas estándar y específicas:
    - periodo_inicio: Fecha de inicio de la quincena (YYYY-MM-DD).
    - periodo_fin: Fecha de fin de la quincena (YYYY-MM-DD).
    - label: Etiqueta identificadora del periodo (ej. 2023-M06_Q1).
    - lat: Latitud de consulta.
    - lon: Longitud de consulta.
    - et_mm: Evapotranspiración real acumulada o promedio estimado para la quincena (mm).
      Nota: Si un valor no está disponible tras agotar ambas fuentes, aparecerá como NaN.
    - fuente: Origen del dato para esa quincena específica ('GF' para MOD16A2GF, 
      'no-GF' para el fallback MOD16A2, o 'sin_dato' si ninguna fuente aportó registro).
"""

from datetime import datetime, date
from pathlib import Path
import ee
import pandas as pd

ee.Initialize()
from period_utils import build_biweekly_periods

def save_evapotranspiration_profile(df, out_prefix="evapotranspiration_profile", output_dir="../databases"):
    """
    Guarda el DataFrame resultante con timestamp en el nombre y estructura estándar.
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    out_path = Path(output_dir) / f"{out_prefix}-v{timestamp}.csv"
    df.to_csv(out_path, index=False)
    print(f"CSV guardado en {out_path} ({df.shape[0]}x{df.shape[1]})")

def get_evapotranspiration_profile(lat, lon, start_date="2016-01-01", end_date=None):
    """
    Extrae el perfil quincenal de evapotranspiración real usando procesamiento server-side.
    """
    # 1. Convertir argumentos de fecha a objetos datetime.date si vienen como string
    if isinstance(start_date, str):
        start_date = datetime.strptime(start_date, "%Y-%m-%d").date()
    
    if end_date is None:
        end_date = datetime.today().date()
    elif isinstance(end_date, str):
        end_date = datetime.strptime(end_date, "%Y-%m-%d").date()

    # 2. Construir la lista de quincenas (devuelve tuplas: label, start, end)
    raw_periods = build_biweekly_periods(start_date, end_date)
    
    # Transformar a ee.List para procesamiento server-side
    ee_periods = ee.List([
        {
            'label': p[0],
            'inicio': p[1].strftime("%Y-%m-%d"),
            'fin': p[2].strftime("%Y-%m-%d")
        } for p in raw_periods
    ])

    point = ee.Geometry.Point([lon, lat])

    # Colecciones MODIS ET (Escala 0.1 para convertir a mm)
    col_gf = ee.ImageCollection('MODIS/061/MOD16A2GF').select('ET').map(
        lambda img: img.multiply(0.1).copyProperties(img, ['system:time_start'])
    )
    col_fallback = ee.ImageCollection('MODIS/061/MOD16A2').select('ET').map(
        lambda img: img.multiply(0.1).copyProperties(img, ['system:time_start'])
    )

    def process_period(period_obj):
        p = ee.Dictionary(period_obj)
        inicio = ee.String(p.get('inicio'))
        fin = ee.String(p.get('fin'))
        label = ee.String(p.get('label'))

        t_start = ee.Date(inicio)
        t_end = ee.Date(fin)

        # Filtrar colecciones
        filtered_gf = col_gf.filterDate(t_start, t_end)
        filtered_fb = col_fallback.filterDate(t_start, t_end)

        has_gf = filtered_gf.size().gt(0)
        has_fb = filtered_fb.size().gt(0)

        # Crear imágenes sumadas o vacías con propiedades de fuente
        img_gf = ee.Algorithms.If(
            has_gf,
            filtered_gf.sum().set('fuente', 'GF'),
            ee.Image.constant(-9999).updateMask(0).set('fuente', 'sin_dato')
        )
        img_gf = ee.Image(img_gf)

        img_fb = ee.Algorithms.If(
            has_fb,
            filtered_fb.sum().set('fuente', 'no-GF'),
            ee.Image.constant(-9999).updateMask(0).set('fuente', 'sin_dato')
        )
        img_fb = ee.Image(img_fb)

        # Selección de imagen basada en la disponibilidad de la fuente primaria (GF)
        final_img = ee.Algorithms.If(has_gf, img_gf, img_fb)
        final_img = ee.Image(final_img)

        # Extraer valor definitivo para el punto de interés
        val_dict = final_img.reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=point,
            scale=500,
            maxPixels=1e9
        )

        et_val = val_dict.get('ET')
        fuente_val = final_img.get('fuente')

        # Verificación robusta en el servidor utilizando ee.Algorithms.If directamente sobre el valor extraído
        final_et = ee.Algorithms.If(et_val, et_val, None)
        final_fuente = ee.Algorithms.If(et_val, fuente_val, 'sin_dato')

        return ee.Dictionary().set('periodo_inicio', inicio)\
                             .set('periodo_fin', fin)\
                             .set('label', label)\
                             .set('et_mm', final_et)\
                             .set('fuente', final_fuente)\
                             .set('lat', lat)\
                             .set('lon', lon)

    # Mapeo server-side de todos los periodos en GEE (UNA SOLA llamada .getInfo() al final)
    print(f"📥 Procesando extracciones quincenales de ET desde 2016 para lat: {lat}, lon: {lon}...")
    results_ee_list = ee_list_to_python(ee_periods.map(process_period).getInfo())
    
    df = pd.DataFrame(results_ee_list)

    # Identificar y reportar quincenas sin datos explícitamente por consola
    missing_data = df[df['et_mm'].isna()]
    if not missing_data.empty:
        print(f"\n⚠️ Aviso: Se encontraron {len(missing_data)} quincenas sin datos disponibles en MODIS ET:")
        for _, row in missing_data.iterrows():
            print(f"   - Periodo: {row['periodo_inicio']} a {row['periodo_fin']} ({row['label']}) -> Fuente: {row['fuente']}")
    else:
        print("\n✅ Todas las quincenas procesadas obtuvieron datos exitosamente.")

    return df

def ee_list_to_python(ee_list):
    """Convierte una lista de diccionarios de GEE obtenida con .getInfo() a formato nativo de Python."""
    return [dict(item) for item in ee_list]

if __name__ == "__main__":
    # Ejemplo de prueba local para Finca Matanza
    Latitude, Longitude =-19.689669877950884,147.22717515914223
    
    # El Playon         --||     7.4584221918243045,    -73.222052853104
    # Finca Matanza     --||     7.300921,              -73.009794
    # Sugarcane_COL     --||     3.580109040361371,     -76.31299479308868
    # Sugarcane_QLD     --||     -19.689669877950884,   147.22717515914223
    
    df_et = get_evapotranspiration_profile(Latitude, Longitude, start_date="2016-01-01")
    
    print("\nMuestra de las primeras 5 filas del resultado:")
    print(df_et.head())
    
    save_evapotranspiration_profile(df_et, out_prefix="et_biweekly")

/home/wmlegion/miniconda3/envs/agri_land_env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


📥 Procesando extracciones quincenales de ET desde 2016 para lat: -19.689669877950884, lon: 147.22717515914223...

✅ Todas las quincenas procesadas obtuvieron datos exitosamente.

Muestra de las primeras 5 filas del resultado:
   et_mm fuente       label       lat         lon periodo_fin periodo_inicio
0   25.8     GF  2016-01_Q1 -19.68967  147.227175  2016-01-16     2016-01-01
1   40.4     GF  2016-01_Q2 -19.68967  147.227175  2016-02-01     2016-01-16
2   51.7     GF  2016-02_Q1 -19.68967  147.227175  2016-02-16     2016-02-01
3   41.6     GF  2016-02_Q2 -19.68967  147.227175  2016-03-01     2016-02-16
4   45.3     GF  2016-03_Q1 -19.68967  147.227175  2016-03-16     2016-03-01
CSV guardado en ../databases/et_biweekly-v260807155558.csv (254x7)


In [3]:
"""
Extrae la serie de evapotranspiración real (ET) quincenal para un punto,
usando doble fuente en Earth Engine:

1. MODIS/061/MOD16A2GF (Gap-Filled) - fuente primaria, con relleno de
   huecos, pero "year-end gap-filled": el año en curso puede no tener
   NINGÚN dato hasta que cierre.
2. MODIS/061/MOD16A2 (no gap-filled) - fuente de respaldo, casi
   tiempo real, para las quincenas donde la primaria todavía no
   publicó. Sin relleno de huecos, así que puede seguir faltando algún
   composite individual por nubosidad persistente.

CÓMO LEER EL RESULTADO
-----------------------
et_total_mm    Agua total perdida por evaporación + transpiración en
               esa quincena (según la vegetación actual del pixel, no
               necesariamente el cultivo que planeás poner - ver nota
               abajo).
fuente         De qué producto salió el dato: 'GF', 'no-GF', o
               'sin_dato' si ninguna de las dos fuentes tenía registro
               para esa quincena.

USO EN EL BALANCE HÍDRICO: junto con precipitation_profile.py (entrada)
y soil_hydraulics.py (capacidad de almacenamiento), esto cierra el
balance:
    Δalmacenamiento = precipitación - ET - excedente (cuando supera AWC)

LIMITACIÓN IMPORTANTE: ET real refleja la vegetación EXISTENTE en el
pixel hoy, no la del cultivo que planees plantar (que puede consumir
más o menos agua). Sirve como piso orientativo para decidir si hace
falta capacidad de riego instalada, no como dimensionamiento exacto
del sistema.

LIMITACIÓN TÉCNICA (composites): ambos productos vienen en composites
de 8 días, que no calzan exactamente con los bordes de las quincenas
(1-15, 16-fin de mes) -> puede haber un pequeño corrimiento en los
bordes. Para el objetivo de detectar déficit general, esto no afecta
la conclusión.

CITAS (para metodología de tesis):
Running, S., Mu, Q., & Zhao, M. (2021). MODIS/Terra Net
Evapotranspiration Gap-Filled 8-Day L4 Global 500m SIN Grid V061
[Data set]. NASA EOSDIS LP DAAC. https://doi.org/10.5067/MODIS/MOD16A2GF.061
Running, S., Mu, Q., & Zhao, M. (2021). MODIS/Terra Net
Evapotranspiration 8-Day L4 Global 500m SIN Grid V061 [Data set].
NASA EOSDIS LP DAAC. https://doi.org/10.5067/MODIS/MOD16A2.061
"""
from datetime import date, datetime
from pathlib import Path
import pandas as pd
import ee

from test_period_utils import build_biweekly_periods
ee.Initialize()

def get_et_biweekly(lat, lon, start_date="2016-01-01", end_date=None):
    """
    lat, lon: coordenadas del punto
    start_date, end_date: rango de fechas (str "YYYY-MM-DD"); end_date=None -> hoy
    """
    start = datetime.strptime(start_date, "%Y-%m-%d").date()
    end = datetime.strptime(end_date, "%Y-%m-%d").date() if end_date else date.today()

    periods = build_biweekly_periods(start, end)
    print(f"[DEBUG] {len(periods)} quincenas a procesar, desde {start} hasta {end}")

    point = ee.Geometry.Point([lon, lat])

    # ET viene escalada: valor crudo x 0.1 = mm reales acumulados en el composite
    col_gf = (
        ee.ImageCollection('MODIS/061/MOD16A2GF')
        .select('ET')
        .map(lambda img: img.multiply(0.1).copyProperties(img, ['system:time_start']))
    )
    col_fallback = (
        ee.ImageCollection('MODIS/061/MOD16A2')
        .select('ET')
        .map(lambda img: img.multiply(0.1).copyProperties(img, ['system:time_start']))
    )

    ee_periods = ee.List([
        {'label': label, 'start': str(p_start), 'end': str(p_end)}
        for label, p_start, p_end in periods
    ])

    def compute_period(period):
        period = ee.Dictionary(period)
        p_start = ee.Date(period.get('start'))
        p_end = ee.Date(period.get('end'))

        filtered_gf = col_gf.filterDate(p_start, p_end)
        filtered_fb = col_fallback.filterDate(p_start, p_end)

        has_gf = filtered_gf.size().gt(0)
        has_fb = filtered_fb.size().gt(0)

        img_gf = ee.Image(ee.Algorithms.If(
            has_gf,
            filtered_gf.sum().rename('et_total_mm').set('fuente', 'GF'),
            ee.Image.constant(0).rename('et_total_mm').selfMask().set('fuente', 'sin_dato')
        ))
        img_fb = ee.Image(ee.Algorithms.If(
            has_fb,
            filtered_fb.sum().rename('et_total_mm').set('fuente', 'no-GF'),
            ee.Image.constant(0).rename('et_total_mm').selfMask().set('fuente', 'sin_dato')
        ))

        # Prioriza GF; si no hay, usa el fallback (que a su vez puede
        # terminar en 'sin_dato' si tampoco hay datos ahí)
        final_img = ee.Image(ee.Algorithms.If(has_gf, img_gf, img_fb))

        stats = final_img.reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=point,
            scale=500,  # resolución nativa de MOD16
            maxPixels=1e9
        )

        return ee.Feature(
            None,
            stats
            .set('label', period.get('label'))
            .set('periodo_inicio', p_start.format('YYYY-MM-dd'))
            .set('periodo_fin', p_end.advance(-1, 'day').format('YYYY-MM-dd'))
            .set('fuente', final_img.get('fuente'))
        )

    features = ee.FeatureCollection(ee_periods.map(compute_period))
    result = features.getInfo()  # única llamada de red para todos los periodos

    rows = []
    for f in result['features']:
        props = f['properties']
        rows.append({
            'periodo_inicio': props.get('periodo_inicio'),
            'periodo_fin': props.get('periodo_fin'),
            'label': props.get('label'),
            # 'lat': lat,
            # 'lon': lon,
            'et_total_mm': round(props.get('et_total_mm'), 2) if props.get('et_total_mm') is not None else None,
            'fuente': props.get('fuente'),
        })

    df = pd.DataFrame(rows)
    df['periodo_inicio'] = pd.to_datetime(df['periodo_inicio'])
    df = df.sort_values('periodo_inicio').reset_index(drop=True)

    filas_nulas = df['et_total_mm'].isna().sum()
    if filas_nulas > 0:
        primeras_nulas = df[df['et_total_mm'].isna()]['label'].tolist()
        print(f"[AVISO] {filas_nulas} quincena(s) sin datos en ninguna fuente "
              f"(GF ni no-GF): {primeras_nulas}")

    return df


def save_et_profile(df, out_prefix="et_biweekly", output_dir="../databases"):
    """
    Guarda la serie de evapotranspiración con timestamp en el nombre:
    {out_prefix}-vYYMMDDHHMMSS.csv (mismo patrón que el resto del pipeline)
    """
    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    df.to_csv(out_path, index=False)
    print(f"CSV guardado en {out_path} ({df.shape[0]}x{df.shape[1]})")
    return out_path


if __name__ == "__main__":
    # Sugarcane_QLD
    LAT = -19.689669877950884
    LON = 147.22717515914223

    df = get_et_biweekly(LAT, LON, start_date="2016-01-01")
    out_path = save_et_profile(df)
    print(df.head(10))

[DEBUG] 254 quincenas a procesar, desde 2016-01-01 hasta 2026-08-12
CSV guardado en ../databases/et_biweekly-v260812003409.csv (254x5)
  periodo_inicio periodo_fin       label  et_total_mm fuente
0     2016-01-01  2016-01-15  2016-01_Q1         25.8     GF
1     2016-01-16  2016-01-31  2016-01_Q2         40.4     GF
2     2016-02-01  2016-02-15  2016-02_Q1         51.7     GF
3     2016-02-16  2016-02-29  2016-02_Q2         41.6     GF
4     2016-03-01  2016-03-15  2016-03_Q1         45.3     GF
5     2016-03-16  2016-03-31  2016-03_Q2         40.1     GF
6     2016-04-01  2016-04-15  2016-04_Q1         29.0     GF
7     2016-04-16  2016-04-30  2016-04_Q2         25.5     GF
8     2016-05-01  2016-05-15  2016-05_Q1         11.0     GF
9     2016-05-16  2016-05-31  2016-05_Q2         25.1     GF
